In [2]:
from pathlib import Path
import json
import re
import sys
import importlib.metadata as md
import numpy as np

ROOT = Path("/home/sagemaker-user/complaintiq")
INDEX = ROOT / "index"

print("=" * 65)
print("1. PYTHON ENVIRONMENT")
print("=" * 65)
print("Python:", sys.version)

for package in [
    "boto3", "botocore", "numpy", "pandas", "faiss-cpu",
    "rank-bm25", "duckdb", "langgraph", "langchain",
    "langchain-aws", "streamlit", "sentence-transformers"
]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n" + "=" * 65)
print("2. RETRIEVAL INDEX VALIDATION")
print("=" * 65)

for name in ["reg_vectors.npy", "case_vectors.npy"]:
    path = INDEX / name
    print(f"\n{name}")

    if not path.exists():
        print("MISSING")
        continue

    try:
        arr = np.load(path, mmap_mode="r")
        print("Shape:", arr.shape)
        print("Data type:", arr.dtype)
        print("Finite values:", bool(np.isfinite(arr).all()))
    except Exception as e:
        print("ERROR:", repr(e))

for name in ["reg_chunks.jsonl", "case_chunks.jsonl"]:
    path = INDEX / name
    print(f"\n{name}")

    if not path.exists():
        print("MISSING")
        continue

    count = 0
    errors = []

    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                json.loads(line)
                count += 1
            except Exception as e:
                errors.append((line_no, str(e)))

    print("Valid records:", count)
    print("Invalid lines:", len(errors))
    if errors:
        print("First errors:", errors[:3])

print("\n" + "=" * 65)
print("3. SMALL DATABASE AND CSV FILES")
print("=" * 65)

for name in [
    "data/complaintiq.duckdb",
    "data/processed/cfpb_index_pool_jan_may_2026.csv",
    "data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv",
    "data/processed/cfpb_holdout_june_2026.csv",
]:
    path = ROOT / name
    print(f"\n{name}")
    print("Exists:", path.exists())
    if path.exists():
        print("Size:", path.stat().st_size, "bytes")
        if path.suffix == ".csv":
            with path.open(encoding="utf-8-sig") as f:
                print("First line:", f.readline().strip()[:300])

print("\n" + "=" * 65)
print("4. PROJECT CONFIGURATION")
print("=" * 65)

config = ROOT / "src" / "config.py"

if config.exists():
    print(config.read_text(encoding="utf-8"))
else:
    print("src/config.py NOT FOUND")

print("\n" + "=" * 65)
print("5. OLD AWS REFERENCES IN NOTEBOOKS AND SOURCE")
print("=" * 65)

pattern = re.compile(
    r"sagemaker-complaintiq-456427397380|456427397380"
)

for folder in ["src", "notebooks", "reports", "docs"]:
    base = ROOT / folder
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix not in [".py", ".ipynb", ".json", ".md", ".txt"]:
            continue

        try:
            content = path.read_text(encoding="utf-8")
        except Exception:
            continue

        for line_no, line in enumerate(content.splitlines(), 1):
            if pattern.search(line):
                print(f"{path.relative_to(ROOT)}:{line_no}: {line.strip()[:250]}")

print("\n" + "=" * 65)
print("VALIDATION COMPLETE — NO FILES MODIFIED")
print("=" * 65)

1. PYTHON ENVIRONMENT
Python: 3.12.15 (main, Oct  6 2026, 23:50:18) [GCC 15.3.0]
boto3: 1.43.117
botocore: 1.43.117
numpy: 1.26.4
pandas: 2.3.3
faiss-cpu: NOT INSTALLED
rank-bm25: NOT INSTALLED
duckdb: 1.5.1
langgraph: NOT INSTALLED
langchain: 0.3.27
langchain-aws: 0.2.35
streamlit: NOT INSTALLED
sentence-transformers: NOT INSTALLED

2. RETRIEVAL INDEX VALIDATION

reg_vectors.npy
Shape: (1784, 1024)
Data type: float32
Finite values: True

case_vectors.npy
Shape: (3000, 1024)
Data type: float32
Finite values: True

reg_chunks.jsonl
Valid records: 1784
Invalid lines: 0

case_chunks.jsonl
Valid records: 3000
Invalid lines: 0

3. SMALL DATABASE AND CSV FILES

data/complaintiq.duckdb
Exists: True
Size: 133 bytes

data/processed/cfpb_index_pool_jan_may_2026.csv
Exists: True
Size: 133 bytes
First line: version https://git-lfs.github.com/spec/v1

data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv
Exists: True
Size: 133 bytes
First line: version https://git-lfs.github.com/spec/v1

dat

In [3]:

from pathlib import Path

ROOT = Path("/home/sagemaker-user/complaintiq")

files = [
    "data/complaintiq.duckdb",
    "data/processed/cfpb_index_pool_jan_may_2026.csv",
    "data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv",
    "data/processed/cfpb_holdout_june_2026.csv",
]

print("=" * 70)
print("DATA FILE CONTENT INSPECTION")
print("=" * 70)

for relative_path in files:
    path = ROOT / relative_path

    print(f"\nFILE: {relative_path}")
    print("Exists:", path.exists())

    if not path.exists():
        continue

    print("Size:", path.stat().st_size, "bytes")

    with path.open("rb") as f:
        header = f.read(250)

    print("First 250 bytes:")
    print(repr(header))

    if b"git-lfs.github.com/spec/v1" in header:
        print("STATUS: GIT LFS POINTER — actual file content missing")
    elif relative_path.endswith(".duckdb"):
        print("STATUS: DATABASE FILE — needs DuckDB integrity check")
    else:
        print("STATUS: NOT IDENTIFIED AS AN LFS POINTER")

print("\n" + "=" * 70)
print("GIT LFS CONFIGURATION")
print("=" * 70)

for name in [".gitattributes", ".gitmodules"]:
    path = ROOT / name
    print(f"\n{name}:")
    if path.exists():
        print(path.read_text(errors="replace")[:5000])
    else:
        print("Not present")

print("\nINSPECTION COMPLETE — NO FILES MODIFIED")


DATA FILE CONTENT INSPECTION

FILE: data/complaintiq.duckdb
Exists: True
Size: 133 bytes
First 250 bytes:
b'version https://git-lfs.github.com/spec/v1\noid sha256:91c5a9e340303dc1f1ab0baf2354caa848f9420ffe66bff0a3ad36d7f1a99882\nsize 95432704\n'
STATUS: GIT LFS POINTER — actual file content missing

FILE: data/processed/cfpb_index_pool_jan_may_2026.csv
Exists: True
Size: 133 bytes
First 250 bytes:
b'version https://git-lfs.github.com/spec/v1\noid sha256:d82713d5a5833d4ce617c411f52bce85922ad5c0727a2504b8a7fc61868be907\nsize 54423475\n'
STATUS: GIT LFS POINTER — actual file content missing

FILE: data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv
Exists: True
Size: 133 bytes
First 250 bytes:
b'version https://git-lfs.github.com/spec/v1\noid sha256:d82713d5a5833d4ce617c411f52bce85922ad5c0727a2504b8a7fc61868be907\nsize 54423475\n'
STATUS: GIT LFS POINTER — actual file content missing

FILE: data/processed/cfpb_holdout_june_2026.csv
Exists: True
Size: 11138095 bytes
First 250 byte

In [4]:
from pathlib import Path
import subprocess
import re

ROOT = Path("/home/sagemaker-user/complaintiq")

def run_git(args):
    result = subprocess.run(
        ["git"] + args,
        cwd=ROOT,
        capture_output=True,
        text=True
    )
    print("Command:", "git", " ".join(args))
    print("Return code:", result.returncode)
    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print(result.stderr.strip())

print("=" * 65)
print("GIT REPOSITORY CHECK")
print("=" * 65)

print("Git directory exists:", (ROOT / ".git").exists())

run_git(["rev-parse", "--is-inside-work-tree"])
run_git(["remote"])

print("\n" + "=" * 65)
print("REMOTE CONFIGURATION (CREDENTIALS REDACTED)")
print("=" * 65)

config = ROOT / ".git" / "config"

if config.exists():
    content = config.read_text(errors="replace")
    content = re.sub(
        r"(https?://)[^/@\s]+@",
        r"\1<REDACTED>@",
        content
    )
    print(content[:6000])
else:
    print(".git/config not found")

print("\n" + "=" * 65)
print("GIT LFS AVAILABILITY")
print("=" * 65)

try:
    result = subprocess.run(
        ["git", "lfs", "version"],
        cwd=ROOT,
        capture_output=True,
        text=True
    )
    print("Return code:", result.returncode)
    print(result.stdout.strip() or result.stderr.strip())
except FileNotFoundError:
    print("Git LFS executable is not installed")

print("\n" + "=" * 65)
print("INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 65)

GIT REPOSITORY CHECK
Git directory exists: False
Command: git rev-parse --is-inside-work-tree
Return code: 128
fatal: not a git repository (or any parent up to mount point /home)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
Command: git remote
Return code: 128
fatal: not a git repository (or any parent up to mount point /home)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).

REMOTE CONFIGURATION (CREDENTIALS REDACTED)
.git/config not found

GIT LFS AVAILABILITY
Return code: 1
git: 'lfs' is not a git command. See 'git --help'.

The most similar command is
	log

INSPECTION COMPLETE — NO FILES MODIFIED


In [5]:
from pathlib import Path
import json
import re

ROOT = Path("/home/sagemaker-user/complaintiq")

print("=" * 75)
print("1. README")
print("=" * 75)

readme = ROOT / "README.md"
if readme.exists():
    print(readme.read_text(errors="replace")[:12000])
else:
    print("README not found")

print("\n" + "=" * 75)
print("2. DATA MANIFEST")
print("=" * 75)

manifest = ROOT / "data" / "manifest.json"
if manifest.exists():
    print(manifest.read_text(errors="replace")[:12000])
else:
    print("Manifest not found")

print("\n" + "=" * 75)
print("3. NOTEBOOK SOURCE REFERENCES")
print("=" * 75)

terms = [
    "cfpb_index_pool_jan_may_2026",
    "cfpb_index_pool_jan_may_2026_deduplicated",
    "complaintiq.duckdb",
    "duckdb.connect",
    "CREATE TABLE",
    "CREATE VIEW",
    "github.com",
    "git-lfs"
]

for notebook_path in sorted((ROOT / "notebooks").glob("*.ipynb")):
    try:
        notebook = json.loads(notebook_path.read_text(errors="replace"))
    except Exception as exc:
        print(f"\nCould not read {notebook_path.name}: {exc}")
        continue

    for index, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        matched = [
            term for term in terms
            if term.lower() in source.lower()
        ]

        if matched:
            print(f"\nNotebook: {notebook_path.name} | Cell: {index}")
            print("Matched:", ", ".join(matched))
            print(source[:3500])

print("\n" + "=" * 75)
print("4. PROJECT DOCUMENTATION REFERENCES")
print("=" * 75)

for folder in ["docs", "src"]:
    base = ROOT / folder
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in {
            ".md", ".py", ".json", ".txt", ".yaml", ".yml"
        }:
            continue

        try:
            content = path.read_text(errors="replace")
        except Exception:
            continue

        matches = [
            line.strip() for line in content.splitlines()
            if re.search(
                r"github\.com|git-lfs|complaintiq\.duckdb|"
                r"cfpb_index_pool_jan_may_2026",
                line,
                re.IGNORECASE
            )
        ]

        if matches:
            print(f"\nFile: {path.relative_to(ROOT)}")
            for line in matches[:15]:
                print(line[:500])

print("\nINSPECTION COMPLETE — NO FILES MODIFIED")

1. README
# ComplaintIQ – AI-Powered Banking Complaints Intelligence Platform

## 1. Project Overview

**ComplaintIQ** is an AI-powered banking complaints intelligence platform designed to analyse consumer complaints, identify complaint categories, retrieve relevant regulatory requirements, investigate similar historical complaints, and generate evidence-grounded responses and analytical insights.

The platform uses Large Language Models (LLMs), Retrieval-Augmented Generation (RAG), structured complaint data, regulatory documents, and an agent-based orchestration workflow.

The objective is to provide a structured intelligence assistant that helps users:

- Understand and classify banking complaints.
- Identify relevant banking products, issues, and sub-issues.
- Retrieve applicable regulatory provisions.
- Analyse complaint trends using structured data.
- Find similar historical complaints.
- Generate evidence-backed regulatory explanations.
- Identify potential consumer vulnerability

In [6]:
from pathlib import Path
import zipfile
import hashlib
import csv
import io

ROOT = Path("/home/sagemaker-user/complaintiq")
ARCHIVE_DIR = ROOT / "data/raw/cfpb_archive"
RAW_DIR = ROOT / "data/raw/cfpb"

archives = [
    "january_february_2026.zip",
    "march_2026.zip",
    "april_2026.zip",
    "may_2026.zip",
    "june_2026.zip",
]

print("=" * 75)
print("1. MONTHLY CFPB ARCHIVE INSPECTION")
print("=" * 75)

for name in archives:
    path = ARCHIVE_DIR / name
    print(f"\nARCHIVE: {name}")
    print("Exists:", path.exists())

    if not path.exists():
        continue

    print("Size:", f"{path.stat().st_size:,}", "bytes")

    try:
        with zipfile.ZipFile(path) as zf:
            members = [
                item for item in zf.infolist()
                if item.filename.lower().endswith(".csv")
            ]

            print("CSV files:", len(members))

            for member in members:
                print("CSV member:", member.filename)
                print("Uncompressed size:",
                      f"{member.file_size:,}", "bytes")

                with zf.open(member) as raw:
                    text = io.TextIOWrapper(
                        raw, encoding="utf-8-sig", newline=""
                    )
                    reader = csv.reader(text)
                    header = next(reader, [])

                print("Column count:", len(header))
                print("Required columns present:", {
                    "Date received",
                    "Product",
                    "Consumer complaint narrative",
                    "Complaint ID"
                }.issubset(set(header)))

    except Exception as exc:
        print("ARCHIVE ERROR:", repr(exc))

print("\n" + "=" * 75)
print("2. RAW CFPB CSV INSPECTION")
print("=" * 75)

for path in sorted(RAW_DIR.glob("*.csv")):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        first_line = f.readline()
        digest.update(first_line)

        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)

    print(f"\nFILE: {path.name}")
    print("Size:", f"{path.stat().st_size:,}", "bytes")
    print("SHA256:", digest.hexdigest())
    print("Header:", first_line.decode("utf-8-sig", errors="replace").strip())

print("\n" + "=" * 75)
print("INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 75)

1. MONTHLY CFPB ARCHIVE INSPECTION

ARCHIVE: january_february_2026.zip
Exists: True
Size: 36,832,036 bytes
CSV files: 1
CSV member: CCDB_Export_15_January_2026_through_February_2026.csv
Uncompressed size: 366,507,123 bytes
Column count: 16
Required columns present: True

ARCHIVE: march_2026.zip
Exists: True
Size: 22,053,643 bytes
CSV files: 1
CSV member: CCDB_Export_16_March_2026.csv
Uncompressed size: 219,594,182 bytes
Column count: 16
Required columns present: True

ARCHIVE: april_2026.zip
Exists: True
Size: 22,963,936 bytes
CSV files: 1
CSV member: CCDB_Export_17_April_2026.csv
Uncompressed size: 223,480,946 bytes
Column count: 16
Required columns present: True

ARCHIVE: may_2026.zip
Exists: True
Size: 21,033,928 bytes
CSV files: 1
CSV member: CCDB_Export_18_May_2026.csv
Uncompressed size: 219,622,001 bytes
Column count: 16
Required columns present: True

ARCHIVE: june_2026.zip
Exists: True
Size: 19,624,767 bytes
CSV files: 1
CSV member: CCDB_Export_19_June_2026.csv
Uncompressed siz

In [7]:

from pathlib import Path
import zipfile
import pandas as pd
import tempfile
import shutil
import hashlib

ROOT = Path("/home/sagemaker-user/complaintiq")
ARCHIVE_DIR = ROOT / "data/raw/cfpb_archive"
EXISTING_HOLDOUT = ROOT / "data/processed/cfpb_holdout_june_2026.csv"

# Create a separate temporary staging directory.
STAGING = Path(tempfile.mkdtemp(prefix="complaintiq_recovery_"))

INDEX_FILE = STAGING / "cfpb_index_pool_jan_may_2026.csv"
DEDUP_FILE = STAGING / "cfpb_index_pool_jan_may_2026_deduplicated.csv"
JUNE_FILE = STAGING / "cfpb_holdout_june_2026.csv"

ARCHIVES = [
    "january_february_2026.zip",
    "march_2026.zip",
    "april_2026.zip",
    "may_2026.zip",
    "june_2026.zip",
]

TARGET_PRODUCTS = {
    "credit card",
    "checking or savings account",
}

DATE_COL = "Date received"
PRODUCT_COL = "Product"
NARRATIVE_COL = "Consumer complaint narrative"
ID_COL = "Complaint ID"

print("=" * 75)
print("COMPLAINTIQ — CFPB DATA RECOVERY")
print("=" * 75)

print("Staging directory:", STAGING)
print("Free disk space:",
      f"{shutil.disk_usage(ROOT).free / (1024**3):.2f} GB")

index_rows = 0
june_rows = 0
index_header_written = False
june_header_written = False

# ------------------------------------------------------------
# 1. PROCESS MONTHLY ARCHIVES
# ------------------------------------------------------------

for archive_name in ARCHIVES:

    archive_path = ARCHIVE_DIR / archive_name

    if not archive_path.exists():
        raise FileNotFoundError(archive_path)

    print(f"\nProcessing: {archive_name}")

    with zipfile.ZipFile(archive_path) as zf:

        csv_names = [
            name for name in zf.namelist()
            if name.lower().endswith(".csv")
        ]

        if len(csv_names) != 1:
            raise RuntimeError(
                f"Expected one CSV in {archive_name}; "
                f"found {len(csv_names)}"
            )

        csv_name = csv_names[0]

        with zf.open(csv_name) as raw:

            for chunk in pd.read_csv(
                raw,
                dtype=str,
                encoding="utf-8-sig",
                keep_default_na=False,
                chunksize=50000,
                low_memory=False,
            ):

                required = {
                    DATE_COL,
                    PRODUCT_COL,
                    NARRATIVE_COL,
                    ID_COL,
                }

                missing = required - set(chunk.columns)

                if missing:
                    raise ValueError(
                        f"{csv_name} missing columns: {sorted(missing)}"
                    )

                dates = pd.to_datetime(
                    chunk[DATE_COL],
                    errors="coerce"
                )

                products = (
                    chunk[PRODUCT_COL]
                    .str.strip()
                    .str.lower()
                )

                narratives = chunk[NARRATIVE_COL].str.strip()
                ids = chunk[ID_COL].str.strip()

                eligible = (
                    products.isin(TARGET_PRODUCTS)
                    & narratives.ne("")
                    & ids.ne("")
                    & dates.notna()
                )

                chunk = chunk.loc[eligible].copy()
                dates = dates.loc[eligible]

                index_mask = (
                    (dates >= "2026-01-01")
                    & (dates < "2026-06-01")
                )

                june_mask = (
                    (dates >= "2026-06-01")
                    & (dates < "2026-07-01")
                )

                index_chunk = chunk.loc[index_mask]
                june_chunk = chunk.loc[june_mask]

                if not index_chunk.empty:
                    index_chunk.to_csv(
                        INDEX_FILE,
                        mode="a",
                        index=False,
                        header=not index_header_written,
                        encoding="utf-8",
                    )

                    index_header_written = True
                    index_rows += len(index_chunk)

                if not june_chunk.empty:
                    june_chunk.to_csv(
                        JUNE_FILE,
                        mode="a",
                        index=False,
                        header=not june_header_written,
                        encoding="utf-8",
                    )

                    june_header_written = True
                    june_rows += len(june_chunk)

    print("Cumulative January–May rows:", f"{index_rows:,}")
    print("Cumulative June rows:", f"{june_rows:,}")

# ------------------------------------------------------------
# 2. CHECK GENERATED FILES
# ------------------------------------------------------------

if not INDEX_FILE.exists() or not JUNE_FILE.exists():
    raise RuntimeError("One or more staging CSV files were not generated.")

print("\n" + "=" * 75)
print("2. RAW DATASET VALIDATION")
print("=" * 75)

index_df = pd.read_csv(
    INDEX_FILE,
    dtype=str,
    keep_default_na=False,
)

june_df = pd.read_csv(
    JUNE_FILE,
    dtype=str,
    keep_default_na=False,
)

print("January–May rows:", f"{len(index_df):,}")
print("June rows:", f"{len(june_df):,}")
print("January–May unique IDs:",
      f"{index_df[ID_COL].nunique():,}")
print("June unique IDs:",
      f"{june_df[ID_COL].nunique():,}")

# ------------------------------------------------------------
# 3. RECREATE DEDUPLICATED DATASET
# ------------------------------------------------------------

duplicate_ids = index_df[ID_COL].duplicated().sum()

dedup_df = index_df.drop_duplicates(
    subset=[ID_COL],
    keep="first"
).copy()

dedup_df.to_csv(
    DEDUP_FILE,
    index=False,
    encoding="utf-8",
)

print("\nDuplicate rows removed:", f"{duplicate_ids:,}")
print("Deduplicated rows:", f"{len(dedup_df):,}")
print("Remaining duplicate IDs:",
      dedup_df[ID_COL].duplicated().sum())

# ------------------------------------------------------------
# 4. VALIDATE DATES AND HOLDOUT SEPARATION
# ------------------------------------------------------------

index_dates = pd.to_datetime(
    index_df[DATE_COL],
    errors="coerce"
)

june_dates = pd.to_datetime(
    june_df[DATE_COL],
    errors="coerce"
)

index_ids = set(dedup_df[ID_COL])
june_ids = set(june_df[ID_COL])

overlap = index_ids.intersection(june_ids)

assert index_dates.notna().all()
assert june_dates.notna().all()
assert index_dates.min() >= pd.Timestamp("2026-01-01")
assert index_dates.max() < pd.Timestamp("2026-06-01")
assert june_dates.min() >= pd.Timestamp("2026-06-01")
assert june_dates.max() < pd.Timestamp("2026-07-01")
assert dedup_df[ID_COL].is_unique
assert len(overlap) == 0

# ------------------------------------------------------------
# 5. COMPARE WITH EXISTING JUNE HOLDOUT
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("5. EXISTING JUNE HOLDOUT COMPARISON")
print("=" * 75)

existing_june = pd.read_csv(
    EXISTING_HOLDOUT,
    dtype=str,
    keep_default_na=False,
)

existing_june_ids = set(existing_june[ID_COL])
generated_june_ids = set(june_df[ID_COL])

print("Existing June rows:", f"{len(existing_june):,}")
print("Generated June rows:", f"{len(june_df):,}")
print("Existing unique IDs:", f"{len(existing_june_ids):,}")
print("Generated unique IDs:", f"{len(generated_june_ids):,}")
print("IDs missing from generated:",
      len(existing_june_ids - generated_june_ids))
print("Additional generated IDs:",
      len(generated_june_ids - existing_june_ids))

# ------------------------------------------------------------
# 6. EXPECTED PROJECT COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("6. EXPECTED COUNT COMPARISON")
print("=" * 75)

print("Expected deduplicated index rows: 31,813")
print("Actual deduplicated index rows:", f"{len(dedup_df):,}")

print("Expected June holdout rows: 6,110")
print("Actual generated June rows:", f"{len(june_df):,}")

print("\nSTAGING FILES:")
for path in [INDEX_FILE, DEDUP_FILE, JUNE_FILE]:
    print(path.name, f"{path.stat().st_size:,} bytes")

print("\nSTAGING SHA256:")
for path in [INDEX_FILE, DEDUP_FILE, JUNE_FILE]:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(path.name, digest)

print("\n" + "=" * 75)
print("RECOVERY STAGING COMPLETE")
print("No original project files were modified.")
print("Do not copy staging files into data/processed yet.")
print("=" * 75)


COMPLAINTIQ — CFPB DATA RECOVERY
Staging directory: /tmp/complaintiq_recovery_q3hmqfew
Free disk space: 4.40 GB

Processing: january_february_2026.zip
Cumulative January–May rows: 11,542
Cumulative June rows: 0

Processing: march_2026.zip
Cumulative January–May rows: 18,354
Cumulative June rows: 0

Processing: april_2026.zip
Cumulative January–May rows: 25,734
Cumulative June rows: 0

Processing: may_2026.zip
Cumulative January–May rows: 31,813
Cumulative June rows: 0

Processing: june_2026.zip
Cumulative January–May rows: 31,813
Cumulative June rows: 6,110

2. RAW DATASET VALIDATION
January–May rows: 31,813
June rows: 6,110
January–May unique IDs: 31,813
June unique IDs: 6,110

Duplicate rows removed: 0
Deduplicated rows: 31,813
Remaining duplicate IDs: 0

5. EXISTING JUNE HOLDOUT COMPARISON
Existing June rows: 6,110
Generated June rows: 6,110
Existing unique IDs: 6,110
Generated unique IDs: 6,110
IDs missing from generated: 0
Additional generated IDs: 0

6. EXPECTED COUNT COMPARISON


In [8]:
from pathlib import Path
import hashlib
import duckdb

ROOT = Path("/home/sagemaker-user/complaintiq")
STAGING = Path("/tmp/complaintiq_recovery_q3hmqfew")

RAW_DIR = ROOT / "data/raw/cfpb"
DB_PATH = STAGING / "complaintiq.duckdb"

EXISTING_JUNE = ROOT / "data/processed/cfpb_holdout_june_2026.csv"
STAGED_JUNE = STAGING / "cfpb_holdout_june_2026.csv"

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=" * 75)
print("1. VERIFY EXISTING JUNE HOLDOUT")
print("=" * 75)

existing_hash = sha256(EXISTING_JUNE)
staged_hash = sha256(STAGED_JUNE)

print("Existing June SHA256:", existing_hash)
print("Staged June SHA256:  ", staged_hash)
print("Exact file match:", existing_hash == staged_hash)

assert existing_hash == staged_hash, (
    "June holdout files differ. Stop before promotion."
)

print("\n" + "=" * 75)
print("2. CREATE STAGING DUCKDB")
print("=" * 75)

if DB_PATH.exists():
    raise FileExistsError(
        f"Staging database already exists: {DB_PATH}. "
        "Do not overwrite it without inspection."
    )

raw_pattern = (RAW_DIR / "*.csv").as_posix()

con = duckdb.connect(str(DB_PATH))

con.execute(f"""
    CREATE OR REPLACE TABLE complaints_raw AS
    SELECT *
    FROM read_csv_auto(
        '{raw_pattern}',
        header = true,
        normalize_names = true,
        all_varchar = true
    )
""")

con.execute("""
CREATE OR REPLACE VIEW v_complaints AS
SELECT
    TRY_CAST(date_received AS DATE) AS date_received,
    product,
    subproduct,
    issue,
    subissue,
    consumer_complaint_narrative AS narrative,
    company_public_response,
    company,
    state,
    zip_code,
    tags,
    submitted_via,
    TRY_CAST(date_sent_to_company AS DATE) AS date_sent_to_company,
    company_response_to_consumer,
    timely_response,
    TRY_CAST(complaint_id AS BIGINT) AS complaint_id
FROM complaints_raw
""")

print("Database created:", DB_PATH)
print("Database size:", f"{DB_PATH.stat().st_size:,}", "bytes")

print("\n" + "=" * 75)
print("3. VALIDATE DATABASE")
print("=" * 75)

raw_count = con.execute(
    "SELECT COUNT(*) FROM complaints_raw"
).fetchone()[0]

view_count = con.execute(
    "SELECT COUNT(*) FROM v_complaints"
).fetchone()[0]

tables = con.execute("SHOW TABLES").fetchdf()
schema = con.execute("DESCRIBE v_complaints").fetchdf()

print("Raw complaint rows:", f"{raw_count:,}")
print("View complaint rows:", f"{view_count:,}")

print("\nTables and views:")
display(tables)

print("\nv_complaints schema:")
display(schema)

print("\nProduct summary:")
display(con.execute("""
    SELECT
        product,
        COUNT(*) AS total_records,
        COUNT(narrative) AS records_with_narrative,
        COUNT(DISTINCT complaint_id) AS unique_complaints,
        MIN(date_received) AS earliest_date,
        MAX(date_received) AS latest_date
    FROM v_complaints
    GROUP BY product
    ORDER BY product
""").fetchdf())

assert raw_count == 37923, f"Unexpected raw count: {raw_count}"
assert view_count == 37923, f"Unexpected view count: {view_count}"

con.close()

print("\n" + "=" * 75)
print("4. REOPEN DATABASE IN READ-ONLY MODE")
print("=" * 75)

ro = duckdb.connect(str(DB_PATH), read_only=True)

readonly_count = ro.execute(
    "SELECT COUNT(*) FROM v_complaints"
).fetchone()[0]

ro.close()

assert readonly_count == 37923

print("Read-only reopen: PASS")
print("Read-only view count:", readonly_count)

print("\n" + "=" * 75)
print("STAGING DATABASE VALIDATION COMPLETE")
print("No original project files were modified.")
print("=" * 75)

1. VERIFY EXISTING JUNE HOLDOUT
Existing June SHA256: fc765cf02aa5d4afd256a4569fd0011b15b9a3349edd60548c84135fea791356
Staged June SHA256:   fc765cf02aa5d4afd256a4569fd0011b15b9a3349edd60548c84135fea791356
Exact file match: True

2. CREATE STAGING DUCKDB
Database created: /tmp/complaintiq_recovery_q3hmqfew/complaintiq.duckdb
Database size: 95,694,848 bytes

3. VALIDATE DATABASE
Raw complaint rows: 37,923
View complaint rows: 37,923

Tables and views:


,name
0,complaints_raw
1,v_complaints



v_complaints schema:


,column_name,column_type,null,key,default,extra
0,date_received,DATE,YES,None,None,None
1,product,VARCHAR,YES,None,None,None
2,subproduct,VARCHAR,YES,None,None,None
3,issue,VARCHAR,YES,None,None,None
4,subissue,VARCHAR,YES,None,None,None
5,narrative,VARCHAR,YES,None,None,None
6,company_public_response,VARCHAR,YES,None,None,None
7,company,VARCHAR,YES,None,None,None
8,state,VARCHAR,YES,None,None,None
9,zip_code,VARCHAR,YES,None,None,None



Product summary:


,product,total_records,records_with_narrative,unique_complaints,earliest_date,latest_date
0,Checking or savings account,19310,19310,19310,2026-01-01,2026-06-30
1,Credit card,18613,18613,18613,2026-01-01,2026-06-30



4. REOPEN DATABASE IN READ-ONLY MODE
Read-only reopen: PASS
Read-only view count: 37923

STAGING DATABASE VALIDATION COMPLETE
No original project files were modified.


In [9]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import hashlib
import os
import duckdb

ROOT = Path("/home/sagemaker-user/complaintiq")
DATA = ROOT / "data"
PROC = DATA / "processed"
STAGING = Path("/tmp/complaintiq_recovery_q3hmqfew")

EXPECTED_INDEX_SHA = (
    "d82713d5a5833d4ce617c411f52bce85922ad5c0727a2504b8a7fc61868be907"
)

EXPECTED_JUNE_SHA = (
    "fc765cf02aa5d4afd256a4569fd0011b15b9a3349edd60548c84135fea791356"
)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

replacements = {
    DATA / "complaintiq.duckdb":
        STAGING / "complaintiq.duckdb",

    PROC / "cfpb_index_pool_jan_may_2026.csv":
        STAGING / "cfpb_index_pool_jan_may_2026.csv",

    PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv":
        STAGING / "cfpb_index_pool_jan_may_2026_deduplicated.csv",
}

# June holdout is deliberately excluded.
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BACKUP_DIR = DATA / "recovery_backups" / timestamp

print("=" * 75)
print("1. PRE-PROMOTION VALIDATION")
print("=" * 75)

for destination, source in replacements.items():
    assert source.is_file(), f"Missing staging file: {source}"
    assert destination.is_file(), f"Missing original file: {destination}"

    print("\nDestination:", destination.relative_to(ROOT))
    print("Staging size:", f"{source.stat().st_size:,}", "bytes")
    print("Original size:", f"{destination.stat().st_size:,}", "bytes")

    with destination.open("rb") as f:
        pointer = f.read(200)

    assert b"git-lfs.github.com/spec/v1" in pointer, (
        f"Original is not an LFS pointer; refusing to overwrite: {destination}"
    )

# Verify known original CSV content.
assert sha256(replacements[PROC / "cfpb_index_pool_jan_may_2026.csv"]) == EXPECTED_INDEX_SHA
assert sha256(replacements[PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv"]) == EXPECTED_INDEX_SHA

assert sha256(
    DATA / "processed/cfpb_holdout_june_2026.csv"
) == EXPECTED_JUNE_SHA

# Validate staged database before promotion.
con = duckdb.connect(
    str(STAGING / "complaintiq.duckdb"),
    read_only=True
)

assert con.execute(
    "SELECT COUNT(*) FROM complaints_raw"
).fetchone()[0] == 37923

assert con.execute(
    "SELECT COUNT(*) FROM v_complaints"
).fetchone()[0] == 37923

con.close()

print("\nAll pre-promotion checks passed.")

# ------------------------------------------------------------
# 2. BACK UP ORIGINAL POINTERS
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("2. BACK UP ORIGINAL POINTER FILES")
print("=" * 75)

BACKUP_DIR.mkdir(parents=True, exist_ok=False)

for destination in replacements:
    backup = BACKUP_DIR / destination.name
    shutil.copy2(destination, backup)

    assert sha256(backup) == sha256(destination)

    print("Backed up:", destination.name)

# ------------------------------------------------------------
# 3. PREPARE TEMPORARY COPIES
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("3. PREPARE RECOVERED FILES")
print("=" * 75)

temporary_files = {}

try:
    for destination, source in replacements.items():
        temporary = destination.with_name(
            destination.name + ".recovery_tmp"
        )

        if temporary.exists():
            raise FileExistsError(temporary)

        shutil.copy2(source, temporary)

        assert temporary.stat().st_size == source.stat().st_size
        assert sha256(temporary) == sha256(source)

        temporary_files[destination] = temporary
        print("Prepared:", destination.name)

    # --------------------------------------------------------
    # 4. ATOMIC REPLACEMENT
    # --------------------------------------------------------

    print("\n" + "=" * 75)
    print("4. PROMOTE RECOVERED FILES")
    print("=" * 75)

    promoted = []

    for destination, temporary in temporary_files.items():
        os.replace(temporary, destination)
        promoted.append(destination)

        print("Promoted:", destination.name)

    # --------------------------------------------------------
    # 5. VERIFY PROMOTED FILES
    # --------------------------------------------------------

    assert sha256(
        PROC / "cfpb_index_pool_jan_may_2026.csv"
    ) == EXPECTED_INDEX_SHA

    assert sha256(
        PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv"
    ) == EXPECTED_INDEX_SHA

    con = duckdb.connect(
        str(DATA / "complaintiq.duckdb"),
        read_only=True
    )

    raw_count = con.execute(
        "SELECT COUNT(*) FROM complaints_raw"
    ).fetchone()[0]

    view_count = con.execute(
        "SELECT COUNT(*) FROM v_complaints"
    ).fetchone()[0]

    con.close()

    assert raw_count == 37923
    assert view_count == 37923

    assert sha256(
        PROC / "cfpb_holdout_june_2026.csv"
    ) == EXPECTED_JUNE_SHA

except Exception:
    print("\nERROR: Promotion or verification failed.")
    print("Attempting rollback from original pointer backups.")

    for destination in replacements:
        backup = BACKUP_DIR / destination.name
        if backup.exists():
            shutil.copy2(backup, destination)

    raise

finally:
    for temporary in temporary_files.values():
        temporary.unlink(missing_ok=True)

print("\n" + "=" * 75)
print("6. RECOVERY SUMMARY")
print("=" * 75)

print("Backup directory:", BACKUP_DIR)

for destination in replacements:
    print(
        "RESTORED |",
        destination.relative_to(ROOT),
        "|",
        f"{destination.stat().st_size:,} bytes",
        "| SHA256:",
        sha256(destination),
    )

print("\nJune holdout: preserved without modification.")
print("Raw CFPB files: preserved.")
print("FAISS indexes: untouched.")
print("Phase 7 results: untouched.")
print("SQLite checkpoint files: untouched.")

print("\nPASS — FILE PROMOTION AND POST-PROMOTION VALIDATION")

1. PRE-PROMOTION VALIDATION

Destination: data/complaintiq.duckdb
Staging size: 95,694,848 bytes
Original size: 133 bytes

Destination: data/processed/cfpb_index_pool_jan_may_2026.csv
Staging size: 54,423,475 bytes
Original size: 133 bytes

Destination: data/processed/cfpb_index_pool_jan_may_2026_deduplicated.csv
Staging size: 54,423,475 bytes
Original size: 133 bytes

All pre-promotion checks passed.

2. BACK UP ORIGINAL POINTER FILES
Backed up: complaintiq.duckdb
Backed up: cfpb_index_pool_jan_may_2026.csv
Backed up: cfpb_index_pool_jan_may_2026_deduplicated.csv

3. PREPARE RECOVERED FILES
Prepared: complaintiq.duckdb
Prepared: cfpb_index_pool_jan_may_2026.csv
Prepared: cfpb_index_pool_jan_may_2026_deduplicated.csv

4. PROMOTE RECOVERED FILES
Promoted: complaintiq.duckdb
Promoted: cfpb_index_pool_jan_may_2026.csv
Promoted: cfpb_index_pool_jan_may_2026_deduplicated.csv

6. RECOVERY SUMMARY
Backup directory: /home/sagemaker-user/complaintiq/data/recovery_backups/20261112T195050Z
RESTOR

In [10]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import duckdb

ROOT = Path("/home/sagemaker-user/complaintiq")
DATA = ROOT / "data"
PROC = DATA / "processed"
INDEX = ROOT / "index"
STATE = ROOT / "state"
EVAL = ROOT / "eval"

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def jsonl_count(path):
    count = 0
    with path.open(encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if line.strip():
                json.loads(line)
                count += 1
    return count

print("=" * 75)
print("1. RESTORED CFPB DATA")
print("=" * 75)

for name in [
    "complaintiq.duckdb",
]:
    path = DATA / name
    print(name, path.stat().st_size, "bytes")

for name in [
    "cfpb_index_pool_jan_may_2026.csv",
    "cfpb_index_pool_jan_may_2026_deduplicated.csv",
    "cfpb_holdout_june_2026.csv",
]:
    path = PROC / name
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    print(
        name,
        "| rows:", f"{len(df):,}",
        "| unique IDs:", f"{df['Complaint ID'].nunique():,}",
        "| SHA256:", sha256(path)
    )

pool = pd.read_csv(
    PROC / "cfpb_index_pool_jan_may_2026_deduplicated.csv",
    dtype=str,
    keep_default_na=False
)
holdout = pd.read_csv(
    PROC / "cfpb_holdout_june_2026.csv",
    dtype=str,
    keep_default_na=False
)

assert len(pool) == 31813
assert len(holdout) == 6110
assert pool["Complaint ID"].is_unique
assert holdout["Complaint ID"].is_unique
assert set(pool["Complaint ID"]).isdisjoint(
    set(holdout["Complaint ID"])
)

print("\nPASS — CFPB dataset counts and separation")

print("\n" + "=" * 75)
print("2. DUCKDB VALIDATION")
print("=" * 75)

db = DATA / "complaintiq.duckdb"
con = duckdb.connect(str(db), read_only=True)

raw_count = con.execute(
    "SELECT COUNT(*) FROM complaints_raw"
).fetchone()[0]

view_count = con.execute(
    "SELECT COUNT(*) FROM v_complaints"
).fetchone()[0]

print("complaints_raw:", f"{raw_count:,}")
print("v_complaints:", f"{view_count:,}")

assert raw_count == 37923
assert view_count == 37923

con.close()
print("PASS — DuckDB read-only validation")

print("\n" + "=" * 75)
print("3. RETRIEVAL INDEX VALIDATION")
print("=" * 75)

index_files = {
    "reg_vectors.npy": INDEX / "reg_vectors.npy",
    "case_vectors.npy": INDEX / "case_vectors.npy",
    "reg_chunks.jsonl": INDEX / "reg_chunks.jsonl",
    "case_chunks.jsonl": INDEX / "case_chunks.jsonl",
}

for name, path in index_files.items():
    assert path.exists(), f"Missing index artifact: {path}"
    print(name, f"{path.stat().st_size:,}", "bytes")

reg_vectors = np.load(
    INDEX / "reg_vectors.npy",
    mmap_mode="r",
    allow_pickle=False
)

case_vectors = np.load(
    INDEX / "case_vectors.npy",
    mmap_mode="r",
    allow_pickle=False
)

reg_chunks = jsonl_count(INDEX / "reg_chunks.jsonl")
case_chunks = jsonl_count(INDEX / "case_chunks.jsonl")

print("\nRegulatory vectors:", reg_vectors.shape)
print("Regulatory chunks:", reg_chunks)
print("Complaint vectors:", case_vectors.shape)
print("Complaint chunks:", case_chunks)

assert reg_vectors.shape == (1784, 1024)
assert case_vectors.shape == (3000, 1024)
assert reg_chunks == 1784
assert case_chunks == 3000

print("PASS — Vector and chunk counts align")

print("\n" + "=" * 75)
print("4. PHASE 7 EVALUATION OUTPUTS")
print("=" * 75)

required_eval_files = [
    EVAL / "gold/G1_triage_gold.jsonl",
    EVAL / "gold/G2_analytics_gold.jsonl",
    EVAL / "gold/G3_regulatory_gold.jsonl",
    EVAL / "results/G1_G3_consolidated_validation.json",
    EVAL / "results/G3_generated_answers_validated.jsonl",
    EVAL / "results/G3_failure_analysis.csv",
    EVAL / "results/G3_final_evaluation_report.json",
    EVAL / "results/G3_manual_answer_review.csv",
    EVAL / "results/phase7_correction_summary.json",
]

for path in required_eval_files:
    assert path.is_file(), f"Missing evaluation artifact: {path}"
    print(
        "PASS |",
        path.relative_to(ROOT),
        "|",
        f"{path.stat().st_size:,}",
        "bytes"
    )

print("\n" + "=" * 75)
print("5. SQLITE STATE FILES — METADATA ONLY")
print("=" * 75)

state_files = [
    "checkpoints.sqlite",
    "checkpoints.sqlite-shm",
    "checkpoints.sqlite-wal",
    "llm_cache.sqlite",
]

for name in state_files:
    path = STATE / name
    print(
        ("PRESENT" if path.exists() else "MISSING"),
        "|", name,
        "|",
        f"{path.stat().st_size:,} bytes" if path.exists()
        else ""
    )

assert (STATE / "checkpoints.sqlite").exists()
assert (STATE / "llm_cache.sqlite").exists()

print("\nNo SQLite database was opened or modified.")

print("\n" + "=" * 75)
print("6. RECOVERY BACKUP")
print("=" * 75)

backup_dir = DATA / "recovery_backups/20261112T195050Z"

for name in [
    "complaintiq.duckdb",
    "cfpb_index_pool_jan_may_2026.csv",
    "cfpb_index_pool_jan_may_2026_deduplicated.csv",
]:
    path = backup_dir / name
    assert path.is_file(), f"Missing original pointer backup: {path}"
    print("PRESERVED |", name, "|", path.stat().st_size, "bytes")

print("\n" + "=" * 75)
print("FINAL INTEGRITY CHECK COMPLETE")
print("No files modified by this validation.")
print("=" * 75)


1. RESTORED CFPB DATA
complaintiq.duckdb 95694848 bytes
cfpb_index_pool_jan_may_2026.csv | rows: 31,813 | unique IDs: 31,813 | SHA256: d82713d5a5833d4ce617c411f52bce85922ad5c0727a2504b8a7fc61868be907
cfpb_index_pool_jan_may_2026_deduplicated.csv | rows: 31,813 | unique IDs: 31,813 | SHA256: d82713d5a5833d4ce617c411f52bce85922ad5c0727a2504b8a7fc61868be907
cfpb_holdout_june_2026.csv | rows: 6,110 | unique IDs: 6,110 | SHA256: fc765cf02aa5d4afd256a4569fd0011b15b9a3349edd60548c84135fea791356

PASS — CFPB dataset counts and separation

2. DUCKDB VALIDATION
complaints_raw: 37,923
v_complaints: 37,923
PASS — DuckDB read-only validation

3. RETRIEVAL INDEX VALIDATION
reg_vectors.npy 7,307,392 bytes
case_vectors.npy 12,288,128 bytes
reg_chunks.jsonl 4,504,700 bytes
case_chunks.jsonl 4,897,915 bytes

Regulatory vectors: (1784, 1024)
Regulatory chunks: 1784
Complaint vectors: (3000, 1024)
Complaint chunks: 3000
PASS — Vector and chunk counts align

4. PHASE 7 EVALUATION OUTPUTS
PASS | eval/gold/G